# Reinforcement Learning — Handwritten Notes Companion Notebook
> *Code implementations of every concept from the scanned handwritten notes*

**Coverage (following your notebook page order):**
1. ε-Greedy Explore–Exploit
2. Optimistic Initial Values
3. UCB1 — Upper Confidence Bounds
4. Incremental Mean Update
5. Thompson Sampling (Beta–Bernoulli)
6. MDP & Credit Assignment
7. Bellman Equations & Value Functions
8. Dynamic Programming — Policy Evaluation & Iteration
9. Value Iteration
10. Monte Carlo Methods
11. TD(0) — Temporal Difference
12. SARSA & Q-Learning
13. Function Approximation (Linear + Neural)

In [ ]:
!pip install -q gymnasium matplotlib numpy scipy
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist
import gymnasium as gym
from collections import defaultdict
import random, math
SEED = 42; np.random.seed(SEED); random.seed(SEED)
print('Setup complete')

---
## 1. ε-Greedy Explore–Exploit
From your notes (page 1–2):
> *ε is a small number; each round we choose to explore or exploit.*
> *If random < ε → explore; strategy will not be optimal (in the short run).*

In [ ]:
class EpsilonGreedyBandit:
    """K-armed bandit solved with epsilon-greedy."""
    def __init__(self, k=10, epsilon=0.1):
        self.k       = k
        self.epsilon = epsilon
        self.q       = np.zeros(k)   # estimated values
        self.n       = np.zeros(k)   # pull counts
        self.true_q  = np.random.randn(k)  # true reward means

    def select_action(self):
        if random.random() < self.epsilon:
            return random.randint(0, self.k - 1)   # explore
        return int(np.argmax(self.q))               # exploit

    def step(self, a):
        reward   = self.true_q[a] + np.random.randn()
        self.n[a] += 1
        # Incremental mean update (from your notes page 4):
        # X_n = X_{n-1} + (1/N)(X_N - X_{n-1})
        self.q[a] += (reward - self.q[a]) / self.n[a]
        return reward

# Compare epsilons
steps = 1000
results = {}
for eps in [0.0, 0.01, 0.1, 0.3]:
    rewards = []
    bandit  = EpsilonGreedyBandit(k=10, epsilon=eps)
    for _ in range(steps):
        a = bandit.select_action()
        rewards.append(bandit.step(a))
    results[eps] = rewards

plt.figure(figsize=(11,4))
for eps, rews in results.items():
    smoothed = np.convolve(rews, np.ones(50)/50, 'valid')
    plt.plot(smoothed, label=f'ε={eps}')
plt.xlabel('Step'); plt.ylabel('Avg reward (50-step window)')
plt.title('ε-Greedy: explore vs exploit trade-off')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## 2. Optimistic Initial Values
From your notes (page 3):
> *True mean ≪ estimated value → forced to explore.*
> *If you don't explore a bandit, the plan will remain high → more exploration.*

In [ ]:
def run_bandit(k=10, steps=1000, epsilon=0.0, init_value=0.0):
    true_q = np.random.randn(k)
    q = np.full(k, init_value, dtype=float)   # optimistic start
    n = np.zeros(k)
    rewards = []
    for _ in range(steps):
        a = np.argmax(q) if random.random() > epsilon else random.randint(0, k-1)
        r = true_q[a] + np.random.randn()
        n[a] += 1
        q[a] += (r - q[a]) / n[a]
        rewards.append(r)
    return rewards

plt.figure(figsize=(11,4))
for init in [0, 2, 5]:
    r = np.convolve(run_bandit(init_value=init), np.ones(50)/50, 'valid')
    plt.plot(r, label=f'Init Q={init} (greedy)')
r = np.convolve(run_bandit(epsilon=0.1, init_value=0), np.ones(50)/50, 'valid')
plt.plot(r, '--', label='ε=0.1, Init=0 (baseline)')
plt.xlabel('Step'); plt.ylabel('Avg reward')
plt.title('Optimistic Initial Values forces early exploration')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## 3. UCB1 — Upper Confidence Bounds
From your notes (page 4):
$$X_{\text{UCB}} = \bar{X}_j + \sqrt{\frac{2\ln N}{N_j}}$$
> *Small sample → large confidence bounds (forces exploration).*
> *Large sample → small bounds (exploitation).*

In [ ]:
class UCB1Bandit:
    def __init__(self, k=10):
        self.k      = k
        self.q      = np.zeros(k)
        self.n      = np.zeros(k)
        self.t      = 0
        self.true_q = np.random.randn(k)

    def select_action(self):
        # Pull each arm at least once
        if self.t < self.k:
            return self.t
        # UCB selection
        ucb = self.q + np.sqrt(2 * np.log(self.t) / (self.n + 1e-9))
        return int(np.argmax(ucb))

    def step(self, a):
        self.t += 1
        r = self.true_q[a] + np.random.randn()
        self.n[a] += 1
        self.q[a] += (r - self.q[a]) / self.n[a]
        return r

steps = 1000
ucb_rewards, eg_rewards = [], []
ucb = UCB1Bandit(k=10)
eg  = EpsilonGreedyBandit(k=10, epsilon=0.1)
# sync true_q
eg.true_q = ucb.true_q.copy()
for _ in range(steps):
    a = ucb.select_action(); ucb_rewards.append(ucb.step(a))
    a = eg.select_action();  eg_rewards.append(eg.step(a))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, rews, label in zip(axes,
    [ucb_rewards, eg_rewards], ['UCB1', 'ε-Greedy (ε=0.1)']):
    smoothed = np.convolve(rews, np.ones(30)/30, 'valid')
    ax.plot(smoothed, label=label)
    ax.set_title(label); ax.set_xlabel('Step'); ax.set_ylabel('Reward')
    ax.grid(True, alpha=0.3)
plt.suptitle('UCB1 vs ε-Greedy'); plt.tight_layout(); plt.show()

# Visualise the UCB bound shrinking
N_total, ns = 1000, np.arange(1, 200)
bounds = np.sqrt(2 * np.log(N_total) / ns)
plt.figure(figsize=(8, 3))
plt.plot(ns, bounds, color='steelblue')
plt.fill_between(ns, 0, bounds, alpha=0.2)
plt.xlabel('N_j (pulls of arm j)'); plt.ylabel('UCB bonus')
plt.title(f'UCB bonus = sqrt(2·ln({N_total}) / N_j) → shrinks with experience')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## 4. Incremental Mean Update
From your notes (page 5):
$$\bar{X}_n = \bar{X}_{n-1} + \frac{1}{N}(X_N - \bar{X}_{n-1})$$
> *Storing all history is expensive. This is the O(1) incremental solution.*

In [ ]:
def incremental_mean_demo(values):
    """Shows equivalence of batch mean vs incremental mean."""
    inc_mean = 0.0
    running  = []
    for n, x in enumerate(values, 1):
        inc_mean = inc_mean + (x - inc_mean) / n   # your formula
        running.append(inc_mean)
    return running

data   = np.random.randn(200) + 3.0   # true mean = 3
inc    = incremental_mean_demo(data)
batch  = [data[:i].mean() for i in range(1, len(data)+1)]

plt.figure(figsize=(10, 3))
plt.plot(batch, 'b-', alpha=0.5, label='Batch mean')
plt.plot(inc,   'r--', linewidth=2, label='Incremental mean')
plt.axhline(3, color='black', linestyle=':', label='True mean=3')
plt.xlabel('Samples seen'); plt.ylabel('Estimate')
plt.title('Incremental mean == Batch mean (O(1) memory)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f'Max difference: {max(abs(b-i) for b,i in zip(batch,inc)):.2e}  (should be ~0)')

---
## 5. Thompson Sampling — Beta–Bernoulli
From your notes (pages 6–8):
$$\text{Beta}(\alpha, \beta) \propto \theta^{\alpha-1}(1-\theta)^{\beta-1}$$
$$\alpha' = \alpha + \#\text{successes}, \quad \beta' = \beta + \#\text{failures}$$
> *Conjugate prior: posterior is still Beta — no integration needed.*

In [ ]:
class ThompsonSamplingBandit:
    def __init__(self, k=10, true_probs=None):
        self.k          = k
        self.true_probs = true_probs or np.random.uniform(0.1, 0.9, k)
        self.alpha      = np.ones(k)   # prior: Beta(1,1) = Uniform
        self.beta       = np.ones(k)

    def select_action(self):
        # Sample theta from each arm's posterior
        samples = np.random.beta(self.alpha, self.beta)
        return int(np.argmax(samples))

    def step(self, a):
        reward = int(np.random.rand() < self.true_probs[a])   # Bernoulli
        self.alpha[a] += reward
        self.beta[a]  += 1 - reward
        return reward

k    = 5
true = np.array([0.2, 0.4, 0.6, 0.75, 0.5])
ts   = ThompsonSamplingBandit(k=k, true_probs=true)

for _ in range(500):
    a = ts.select_action(); ts.step(a)

# Visualise posteriors after 500 steps
theta = np.linspace(0, 1, 300)
fig, axes = plt.subplots(1, k, figsize=(14, 3), sharey=True)
colors = plt.cm.tab10(np.linspace(0, 1, k))
for i, ax in enumerate(axes):
    pdf = beta_dist.pdf(theta, ts.alpha[i], ts.beta[i])
    ax.plot(theta, pdf, color=colors[i])
    ax.fill_between(theta, pdf, alpha=0.3, color=colors[i])
    ax.axvline(true[i], color='red', linestyle='--', label=f'True={true[i]}')
    ax.set_title(f'Arm {i}\nα={ts.alpha[i]:.0f} β={ts.beta[i]:.0f}')
    ax.legend(fontsize=7)
plt.suptitle('Thompson Sampling: Beta posteriors after 500 steps', fontsize=12)
plt.tight_layout(); plt.show()

---
## 6. Bellman Equations & Value Functions
From your notes (pages 12–13):
$$V_\pi(s) = \sum_a \Pi(a|s)\sum_{s',r} P(s',r|s,a)\bigl[r + \delta V(s')\bigr]$$
$$Q^*(s,a) = \mathbb{E}\bigl[R_{t+1} + \gamma \max_{a'} Q^*(s',a') \mid s,a\bigr]$$

In [ ]:
# Simple MDP: states {start, mid, end}, verify Bellman holds
# Transition: start --(a=go)--> mid (r=0), mid --(a=go)--> end (r=1)
gamma = 0.9

# Manual Bellman computation
V_end   = 0.0          # terminal
V_mid   = 0 + gamma * V_end + 1    # r=1 from mid->end
V_start = 0 + gamma * V_mid        # r=0 from start->mid

print('--- Bellman Equation Manual Verification ---')
print(f'V(end)   = {V_end}')
print(f'V(mid)   = r + γ·V(end)   = 1 + {gamma}·{V_end} = {V_mid}')
print(f'V(start) = r + γ·V(mid)   = 0 + {gamma}·{V_mid} = {V_start}')

# Discount visualisation (from your notes page 2)
gammas = [0.5, 0.9, 0.99]
steps  = np.arange(20)
plt.figure(figsize=(10, 3))
for g in gammas:
    plt.plot(steps, g**steps, label=f'γ={g}')
plt.xlabel('Steps into future'); plt.ylabel('Discount weight γᵏ')
plt.title('Discounting: how much future reward matters')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## 7. Dynamic Programming — Policy Evaluation
From your notes (page 14–15):
```
WHILE True: Δ = 0
  V(s) = Σ_a π(a|s) Σ_{s',r} P(s',r|s,a)[r + γV(s')]
  Δ = max(Δ, |V_old - V(s)|)
  if Δ < threshold: break
```

In [ ]:
# GridWorld 4x4 — Policy Evaluation
# States: 0..15, terminal: 0 and 15
# Actions: 0=up,1=down,2=left,3=right

GRID = 4
N_STATES  = GRID * GRID
N_ACTIONS = 4
GAMMA     = 1.0   # undiscounted (as in standard gridworld)
THRESHOLD = 1e-6

def step_grid(s, a):
    """Returns (next_state, reward)."""
    if s == 0 or s == N_STATES - 1:
        return s, 0                   # terminal
    row, col = s // GRID, s % GRID
    if   a == 0: row = max(row-1, 0)
    elif a == 1: row = min(row+1, GRID-1)
    elif a == 2: col = max(col-1, 0)
    elif a == 3: col = min(col+1, GRID-1)
    return row * GRID + col, -1       # cost -1 per step

# Uniform random policy: π(a|s) = 0.25
V = np.zeros(N_STATES)
history = []
for iteration in range(1000):
    delta = 0
    V_new = V.copy()
    for s in range(N_STATES):
        if s == 0 or s == N_STATES-1:
            continue
        v = 0
        for a in range(N_ACTIONS):
            s2, r = step_grid(s, a)
            v += 0.25 * (r + GAMMA * V[s2])   # Bellman update
        delta   = max(delta, abs(v - V[s]))
        V_new[s] = v
    V = V_new
    history.append(delta)
    if delta < THRESHOLD:
        print(f'Converged in {iteration+1} iterations')
        break

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(V.reshape(GRID, GRID), cmap='RdYlGn')
plt.colorbar()
for i in range(GRID):
    for j in range(GRID):
        plt.text(j, i, f'{V[i*GRID+j]:.1f}', ha='center', va='center', fontsize=9)
plt.title('V(s) under random policy')
plt.subplot(1, 2, 2)
plt.semilogy(history)
plt.xlabel('Iteration'); plt.ylabel('Max |ΔV| (log)')
plt.title('Convergence of Policy Evaluation')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## 8. Policy Iteration
From your notes (pages 15–16):
> *Alternating between policy evaluation and policy improvement.*
> *Find a ∈ A s.t. Q_π(s,a) > Q_π(s) ← optimal policy.*

In [ ]:
def policy_evaluation(policy, gamma=1.0, thr=1e-6):
    V = np.zeros(N_STATES)
    while True:
        delta = 0
        for s in range(N_STATES):
            if s in (0, N_STATES-1): continue
            a = policy[s]
            s2, r = step_grid(s, a)
            v_new  = r + gamma * V[s2]
            delta  = max(delta, abs(v_new - V[s]))
            V[s]   = v_new
        if delta < thr: break
    return V

def policy_improvement(V, gamma=1.0):
    policy = np.zeros(N_STATES, dtype=int)
    for s in range(N_STATES):
        if s in (0, N_STATES-1): continue
        q_vals = []
        for a in range(N_ACTIONS):
            s2, r = step_grid(s, a)
            q_vals.append(r + gamma * V[s2])
        policy[s] = int(np.argmax(q_vals))
    return policy

# Policy Iteration
policy = np.zeros(N_STATES, dtype=int)  # start: all 'up'
for pi_iter in range(100):
    V      = policy_evaluation(policy)
    new_pi = policy_improvement(V)
    if np.all(new_pi == policy):
        print(f'Policy converged after {pi_iter+1} policy iterations')
        break
    policy = new_pi

action_symbols = ['^', 'v', '<', '>']
plt.figure(figsize=(5,5))
plt.imshow(V.reshape(GRID,GRID), cmap='RdYlGn')
plt.colorbar()
for i in range(GRID):
    for j in range(GRID):
        s = i*GRID+j
        sym = 'T' if s in (0, N_STATES-1) else action_symbols[policy[s]]
        plt.text(j, i, sym, ha='center', va='center', fontsize=14, fontweight='bold')
plt.title('Optimal Policy (Policy Iteration) on 4×4 GridWorld')
plt.tight_layout(); plt.show()

---
## 9. Value Iteration
From your notes (page 17):
$$V_{k+1}(s) = \max_a \sum_{s',r} P(s',r|s,a)\bigl[r + \delta V_k(s)\bigr]$$
> *No need to do full policy evaluation. Greedily update V directly.*

In [ ]:
V = np.zeros(N_STATES)
vi_history = []

for vi_iter in range(10000):
    delta = 0
    for s in range(N_STATES):
        if s in (0, N_STATES-1): continue
        q_vals = []
        for a in range(N_ACTIONS):
            s2, r = step_grid(s, a)
            q_vals.append(r + GAMMA * V[s2])
        v_new  = max(q_vals)
        delta  = max(delta, abs(v_new - V[s]))
        V[s]   = v_new
    vi_history.append(delta)
    if delta < THRESHOLD:
        print(f'Value Iteration converged in {vi_iter+1} sweeps')
        break

# Extract greedy policy
vi_policy = policy_improvement(V)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(V.reshape(GRID,GRID), cmap='RdYlGn')
axes[0].set_title('V*(s) from Value Iteration')
for i in range(GRID):
    for j in range(GRID):
        s = i*GRID+j
        sym = 'T' if s in (0, N_STATES-1) else action_symbols[vi_policy[s]]
        axes[0].text(j, i, sym, ha='center', va='center', fontsize=14)
axes[1].semilogy(vi_history)
axes[1].set_xlabel('Sweep'); axes[1].set_ylabel('Max |ΔV|')
axes[1].set_title('Value Iteration Convergence')
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 10. Monte Carlo Methods
From your notes (pages 18–19):
> *Every visit method: values only updated for visited states.*
> *Use Exploring Starts to cover all (s,a) pairs.*

In [ ]:
# Monte Carlo First-Visit on FrozenLake
env = gym.make('FrozenLake-v1', is_slippery=False)

def mc_episode(env, policy, max_steps=200):
    """Generate one episode following policy. Returns list of (s,a,r)."""
    traj = []
    s, _ = env.reset()
    for _ in range(max_steps):
        a = policy[s]
        s2, r, term, trunc, _ = env.step(a)
        traj.append((s, a, r))
        s = s2
        if term or trunc: break
    return traj

# Random policy to start
n_s = env.observation_space.n
n_a = env.action_space.n
policy = np.random.randint(0, n_a, n_s)

Q_mc   = np.zeros((n_s, n_a))
N_mc   = np.zeros((n_s, n_a))
GAMMA  = 0.99
wins   = []

for ep in range(5000):
    # Exploring start: random initial state & action
    traj = mc_episode(env, policy)
    wins.append(traj[-1][2])   # reward at terminal
    G = 0
    visited = set()
    for t in reversed(range(len(traj))):
        s, a, r = traj[t]
        G = r + GAMMA * G
        if (s, a) not in visited:     # First-visit MC
            visited.add((s, a))
            N_mc[s, a] += 1
            Q_mc[s, a] += (G - Q_mc[s, a]) / N_mc[s, a]
    # Greedy policy improvement
    policy = np.argmax(Q_mc, axis=1)

smoothed = np.convolve(wins, np.ones(200)/200, 'valid')
plt.figure(figsize=(10, 3))
plt.plot(smoothed)
plt.xlabel('Episode'); plt.ylabel('Win rate (200-ep window)')
plt.title('Monte Carlo Control on FrozenLake (First-Visit)')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f'Final policy win rate: {np.mean(wins[-500:]):.1%}')

---
## 11. TD(0) — Temporal Difference Prediction
From your notes (pages 20–21):
$$\theta = \theta + \alpha\bigl(r + \delta\hat{V}(s') - \hat{V}(s,\theta)\bigr)\nabla_\theta\hat{V}(s,\theta)$$
> *G ≠ TD(0): G = r + γV(s') — called semi-gradient descent.*

In [ ]:
# TD(0) Prediction on FrozenLake
env    = gym.make('FrozenLake-v1', is_slippery=False)
V_td   = np.zeros(env.observation_space.n)
ALPHA  = 0.1
GAMMA  = 0.99
errors = []

for ep in range(3000):
    s, _ = env.reset()
    ep_err = []
    for _ in range(200):
        a = env.action_space.sample()
        s2, r, term, trunc, _ = env.step(a)
        # TD(0) update: V(s) <- V(s) + alpha*(r + gamma*V(s') - V(s))
        td_target = r + GAMMA * V_td[s2] * (not term)
        td_error  = td_target - V_td[s]
        V_td[s]  += ALPHA * td_error
        ep_err.append(abs(td_error))
        s = s2
        if term or trunc: break
    errors.append(np.mean(ep_err))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(V_td.reshape(4,4), cmap='YlGn')
axes[0].set_title('V(s) learned by TD(0)')
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f'{V_td[i*4+j]:.2f}', ha='center', va='center', fontsize=9)
smoothed = np.convolve(errors, np.ones(100)/100, 'valid')
axes[1].plot(smoothed)
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Mean |TD error|')
axes[1].set_title('TD Error converges to 0')
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 12. SARSA & Q-Learning
From your notes (pages 21–23):
> *SARSA: on-policy. Q-Learning: off-policy (uses max over next actions).*
> *SA ≡ ID × x/(s,a) = workaround x → unique x*

In [ ]:
def run_td_control(env, method='qlearning', n_episodes=5000,
                   alpha=0.1, gamma=0.99, eps=0.1):
    n_s = env.observation_space.n
    n_a = env.action_space.n
    Q   = np.zeros((n_s, n_a))
    wins = []

    for ep in range(n_episodes):
        s, _ = env.reset()
        # ε-greedy action selection
        def choose(state):
            if random.random() < eps: return env.action_space.sample()
            return int(np.argmax(Q[state]))

        a = choose(s)
        done = False
        while not done:
            s2, r, term, trunc, _ = env.step(a)
            done = term or trunc
            a2   = choose(s2)

            if method == 'sarsa':
                # SARSA: on-policy — uses actual next action a2
                target = r + gamma * Q[s2, a2] * (not term)
            else:
                # Q-Learning: off-policy — uses max over next actions
                target = r + gamma * np.max(Q[s2]) * (not term)

            Q[s, a] += alpha * (target - Q[s, a])
            s, a = s2, a2
        wins.append(r)
    return Q, wins

env = gym.make('FrozenLake-v1', is_slippery=False)
Q_sarsa, wins_sarsa = run_td_control(env, method='sarsa')
Q_ql,    wins_ql    = run_td_control(env, method='qlearning')

plt.figure(figsize=(11, 4))
for wins, label in [(wins_sarsa, 'SARSA'), (wins_ql, 'Q-Learning')]:
    smoothed = np.convolve(wins, np.ones(200)/200, 'valid')
    plt.plot(smoothed, label=label)
plt.xlabel('Episode'); plt.ylabel('Win rate (200-ep)')
plt.title('SARSA vs Q-Learning on FrozenLake')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## 13. Function Approximation — Linear & Neural
From your notes (pages 24–27):
$$\theta = \theta + \alpha(G - \hat{V}(s,\theta))\nabla_\theta\hat{V}(s,\theta)$$
> *Linear model not always sufficient → Taylor expansion → neural net.*
> *The linear model is not always increasing — solution: feature polynomials.*

In [ ]:
import torch
import torch.nn as nn

# Linear function approximation for V(s)
# Feature map: one-hot encoding of state
class LinearValueApprox:
    def __init__(self, n_states):
        self.theta = np.zeros(n_states)

    def features(self, s):
        x = np.zeros(len(self.theta))
        x[s] = 1.0
        return x

    def predict(self, s):
        return float(self.theta @ self.features(s))

    def update(self, s, target, alpha=0.05):
        x     = self.features(s)
        error = target - self.predict(s)
        # θ = θ + α(G - V̂(s,θ)) ∇_θ V̂  ← gradient for linear is just x
        self.theta += alpha * error * x

# Train with TD(0) targets on FrozenLake
env  = gym.make('FrozenLake-v1', is_slippery=False)
model = LinearValueApprox(n_states=env.observation_space.n)
GAMMA, ALPHA = 0.99, 0.05

for ep in range(5000):
    s, _ = env.reset()
    for _ in range(200):
        a = env.action_space.sample()
        s2, r, term, trunc, _ = env.step(a)
        # Semi-gradient TD(0) target
        target = r + GAMMA * model.predict(s2) * (not term)
        model.update(s, target, alpha=ALPHA)
        s = s2
        if term or trunc: break

V_linear = model.theta.reshape(4,4)
plt.figure(figsize=(5, 4))
plt.imshow(V_linear, cmap='YlGn')
plt.colorbar()
for i in range(4):
    for j in range(4):
        plt.text(j, i, f'{V_linear[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.title('Linear Function Approx V̂(s,θ) on FrozenLake')
plt.tight_layout(); plt.show()

In [ ]:
# Neural network V(s) approximation (from your DRL diagram, page 27)
class NeuralValueNet(nn.Module):
    def __init__(self, n_states):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_states, 64), nn.ReLU(),
            nn.Linear(64, 32),       nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

n_s     = env.observation_space.n
net     = NeuralValueNet(n_s)
optim   = torch.optim.Adam(net.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

def one_hot(s, n): 
    x = torch.zeros(n); x[s] = 1.0; return x

losses = []
for ep in range(5000):
    s, _ = env.reset()
    ep_loss = []
    for _ in range(200):
        a = env.action_space.sample()
        s2, r, term, trunc, _ = env.step(a)
        with torch.no_grad():
            target = r + GAMMA * net(one_hot(s2, n_s)) * (not term)
        pred = net(one_hot(s, n_s))
        loss = loss_fn(pred, target)
        optim.zero_grad(); loss.backward(); optim.step()
        ep_loss.append(loss.item())
        s = s2
        if term or trunc: break
    losses.append(np.mean(ep_loss))

V_nn = np.array([net(one_hot(s, n_s)).item() for s in range(n_s)]).reshape(4, 4)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(V_nn, cmap='YlGn'); axes[0].set_title('Neural Net V̂(s,θ)')
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f'{V_nn[i,j]:.2f}', ha='center', va='center', fontsize=9)
smoothed = np.convolve(losses, np.ones(100)/100, 'valid')
axes[1].semilogy(smoothed); axes[1].set_title('Neural Net Training Loss')
axes[1].set_xlabel('Episode'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print('Done — all 13 sections implemented!')